In [12]:
import pandas as pd
import numpy as np

In [13]:
# ═══════════════════════════════════════════════════════════════
# Staff School by education level (Staff_18_19 / 19_20 / 20_21)
# ═══════════════════════════════════════════════════════════════
files = {
    "2018-2019": "public_schools_2018-2019.xlsx",
    "2019-2020": "public_schools_2019-2020.xlsx",
    "2020-2021": "public_schools_2020-2021.xlsx",
}
 
sheets = "School Staff by Education Level"   
output_file   = "school_staff_by_education_level_cleaned.xlsx"

def find_sheet(xl: pd.ExcelFile, keyword: str) -> str:
    """Return the first sheet name whose lowercase contains keyword."""
    for name in xl.sheet_names:
        if keyword in name.lower():
            return name
    raise ValueError(
        f"No sheet matching '{keyword}' found.\nAvailable: {xl.sheet_names}")

In [14]:
def build_column_names(group_row, sub_row) -> list[str]:
    group_row = list(group_row)
    sub_row   = list(sub_row)
current_group = ""
filled_groups = []
for i in filled_groups:
    if pd.notna(i) and str(i).strip():
        current_group = str(i).strip()
    filled_groups.append(current_group)

In [15]:
def clean_sub(val) -> str:
    if pd.isna(val):
        return ""
    return (
        str(val).strip()
        .replace("\n", " ")          
        .replace("Gaduate", "Graduate"))

sub_row = ["", "Graduate", "Post Graduate", "Doctorate", "Unknown"]
cleaned_subs = [clean_sub(v) for v in sub_row]

In [16]:
def build_column_names(group_row, sub_row):
    group_row = list(group_row)
    sub_row   = list(sub_row)

    current_group = ""
    filled_groups = []
    for val in group_row:
        if pd.notna(val) and str(val).strip():
            current_group = str(val).strip()
        filled_groups.append(current_group)

    def clean_sub(val):
        if pd.isna(val):
            return ""
        return (
            str(val).strip()
            .replace("\n", " ")
            .replace("Gaduate", "Graduate")
        )

    cleaned_subs = [clean_sub(v) for v in sub_row]

    columns = []
    for i, (grp, sub) in enumerate(zip(filled_groups, cleaned_subs)):
        if i == 0:
            columns.append("Province")
        elif sub:
            grp_short = (
                grp
                .replace("Education Level of ", "")
                .replace(" without Pedagogy Training", "_No_Pedagogy")
            )
            columns.append(f"{grp_short}__{sub}")
        else:
            columns.append(f"col_{i}")
    return columns

In [17]:
def clean_sheet(path: str, year: str) -> pd.DataFrame:
    xl    = pd.ExcelFile(path)
    sheet = find_sheet(xl, sheets)
    raw   = pd.read_excel(path, sheet_name=sheet, header=None)
 
    print(f"  Reading '{sheet}' from {path}  →  raw shape: {raw.shape}")

In [18]:
def clean_sheet(path, year):
    xl    = pd.ExcelFile(path)
    sheet = find_sheet(xl, "staff by education")
    raw   = pd.read_excel(path, sheet_name=sheet, header=None)

    first_cell = str(raw.iloc[0, 0]).strip()
    has_title  = first_cell.lower().startswith("table")
    hdr_start  = 1 if has_title else 0

    columns = build_column_names(raw.iloc[hdr_start], raw.iloc[hdr_start + 1])

    data = raw.iloc[hdr_start + 2:].copy()
    data.columns = columns
    data = data.reset_index(drop=True)

    data = data[data["Province"].notna()]
    data = data[data["Province"].astype(str).str.strip() != ""]
    data = data[data["Province"].astype(str).str.strip() != "nan"]

    data["Province"] = (
        data["Province"]
        .astype(str)
        .str.strip()
        .str.replace(r"^-\s*", "", regex=True)
        .str.strip()
    )

    num_cols = [c for c in data.columns if c != "Province"]
    for col in num_cols:
        data[col] = pd.to_numeric(data[col], errors="coerce").astype("Int64")

    data = data.dropna(how="all").reset_index(drop=True)
    data.insert(0, "Academic_Year", year)
    return data

In [19]:
provinces = [
    "Banteay Meanchey", "Battambang", "Kampong Cham", "Kampong Chhnang",
    "Kampong Speu", "Kampong Thom", "Kampot", "Kandal", "Kep", "Koh Kong",
    "Kratie", "Mondul Kiri", "Otdar Meanchey", "Pailin", "Phnom Penh",
    "Preah Sihanouk", "Preah Vihear", "Prey Veng", "Pursat", "Ratanak Kiri",
    "Siemreap", "Stung Treng", "Svay Rieng", "Takeo", "Tbaung Khmum",
    "Whole Kingdom", "Urban Area", "Rural Area",
]
 
def validate(year: str, df: pd.DataFrame) -> None:
    print(f"\n  Validation [{year}]")
 
    # Province check
    found   = set(df["Province"].tolist())
    missing = [p for p in provinces if p not in found]
    extra   = [p for p in found if p not in provinces]
    if missing:
        print(f"Missing provinces : {missing}")
    else:
        print(f"All 28 provinces present")
    if extra:
        print(f" Extra rows        : {extra}")
    bad_cols = [c for c in df.columns if "\n" in c or "Gaduate" in c]
    if bad_cols:
        print(f"Bad column names : {bad_cols}")
    else:
        print(f"Column names clean (no newlines / typos)")
 
    # Null count
    num_cols = [c for c in df.columns if c not in ("Academic_Year", "Province")]
    total_nulls = df[num_cols].isna().sum().sum()
    print(f"NaN in numeric cols: {total_nulls}")

In [21]:
def main():
    print("=" * 60)
    print("  School Staff by Education Level Cleaning Pipeline")
    print("=" * 60)

    cleaned = {}
 
    for year, path in files.items():
        print(f"\n[{year}]")
        df = clean_sheet(path, year)
        validate(year, df)
        cleaned[year] = df
 
    # Combine all years
    combined = pd.concat(cleaned.values(), ignore_index=True)
    print(f"\n  Combined shape: {combined.shape}  (all three years stacked)")
 
    # Write to Excel
    print(f"\n  Saving  {output_file}")
    with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
        combined.to_excel(writer, sheet_name="All Years Combined", index=False)
        for year, df in cleaned.items():
            df.to_excel(writer, sheet_name=year, index=False)
 
    # Apply formatting
    format_workbook(output_file)
 
    print("\n" + "=" * 60)
    print(f"  Done!  Output: {output_file}")
    print(f"  Sheets: 'All Years Combined', " +
          ", ".join(f"'{y}'" for y in cleaned))
    print("=" * 60)